# CSIRO Biomass - 5 Fold Inference (Fixed Model Structure)

**変更点**: トレーニングコードと同じモデル構造を使用

In [ ]:
!pip uninstall -y timm -q
!pip install -q --no-deps /kaggle/input/wheels-csiro/timm-1.0.22-py3-none-any.whl

In [ ]:
import os
import random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
from tqdm import tqdm
import gc
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"timm: {timm.__version__}")

In [ ]:
class CFG:
    TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
    DATA_DIR = Path("/kaggle/input/csiro-biomass")
    MODEL_DIR = Path("/kaggle/input/csiro-dinov3-notta")
    IMG_SIZE = 512
    N_FOLDS = 5
    BACKBONE = "vit_huge_plus_patch16_dinov3.lvd1689m"
    BATCH_SIZE = 1
    NUM_WORKERS = 0

In [ ]:
# Load test.csv - これが重要！sample_idを含んでいる
test_df = pd.read_csv(CFG.DATA_DIR / "test.csv")
print(f"test_df columns: {test_df.columns.tolist()}")
print(f"test_df shape: {test_df.shape}")
print(f"\nFirst 10 rows:")
print(test_df.head(10))

# Get unique test images
test_wide = test_df[["image_path"]].drop_duplicates().reset_index(drop=True)
print(f"\nUnique test images: {len(test_wide)}")

In [ ]:
# ============================================================
# MODEL DEFINITION (トレーニングコードと完全に同じ構造)
# ============================================================

class LocalMambaBlock(nn.Module):
    """トレーニングコードと同じ実装"""
    def __init__(self, dim, kernel_size=5, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.dwconv = nn.Conv1d(dim, dim, kernel_size=kernel_size, padding=kernel_size//2, groups=dim)
        self.gate = nn.Linear(dim, dim)
        self.proj = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        shortcut = x
        x = self.norm(x)
        # トレーニングコードと同じ: gate適用後にdwconv
        x = x * torch.sigmoid(self.gate(x))
        x = self.dwconv(x.transpose(1, 2)).transpose(1, 2)
        x = self.proj(x)
        return shortcut + self.drop(x)


class BiomassModel(nn.Module):
    """トレーニングコードと同じ実装"""
    def __init__(self, model_name, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool="")
        nf = self.backbone.num_features
        
        # Gradient checkpointing (推論時は不要だが、構造の一貫性のため)
        if hasattr(self.backbone, "set_grad_checkpointing"):
            self.backbone.set_grad_checkpointing(True)
            print("✅ Gradient Checkpointing enabled")
        
        # Fusion layers
        self.fusion = nn.Sequential(
            LocalMambaBlock(nf, kernel_size=5, dropout=0.1),
            LocalMambaBlock(nf, kernel_size=5, dropout=0.1)
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        
        # Prediction heads (トレーニングコードと同じ)
        self.head_green = nn.Sequential(
            nn.Linear(nf, nf//2), 
            nn.GELU(), 
            nn.Dropout(0.2), 
            nn.Linear(nf//2, 1), 
            nn.Softplus()
        )
        self.head_dead = nn.Sequential(
            nn.Linear(nf, nf//2), 
            nn.GELU(), 
            nn.Dropout(0.2), 
            nn.Linear(nf//2, 1), 
            nn.Softplus()
        )
        self.head_clover = nn.Sequential(
            nn.Linear(nf, nf//2), 
            nn.GELU(), 
            nn.Dropout(0.2), 
            nn.Linear(nf//2, 1), 
            nn.Softplus()
        )

    def forward(self, x):
        left, right = x
        x_l = self.backbone(left)
        x_r = self.backbone(right)
        # トレーニングコードと同じ fusion
        x = self.fusion(torch.cat([x_l, x_r], dim=1))
        x = self.pool(x.transpose(1, 2)).flatten(1)
        
        green = self.head_green(x)
        dead = self.head_dead(x)
        clover = self.head_clover(x)
        gdm = green + clover
        total = green + clover + dead
        
        return torch.cat([green, dead, clover, gdm, total], dim=1)

print("✅ Model defined (same as training code)")

In [ ]:
class TestDataset(Dataset):
    def __init__(self, df, data_dir, transform):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.data_dir / row["image_path"]
        img = Image.open(img_path).convert("RGB")
        w, h = img.size
        left = img.crop((0, 0, w // 2, h))
        right = img.crop((w // 2, 0, w, h))
        left = self.transform(left)
        right = self.transform(right)
        return left, right, row["image_path"]

def collate_fn(batch):
    lefts = torch.stack([b[0] for b in batch])
    rights = torch.stack([b[1] for b in batch])
    paths = [b[2] for b in batch]
    return lefts, rights, paths

test_tfms = T.Compose([
    T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("✅ Dataset defined")

In [ ]:
test_dataset = TestDataset(test_wide, CFG.DATA_DIR, test_tfms)
test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    collate_fn=collate_fn
)

print(f"DataLoader: {len(test_dataset)} images")

FOLD_WEIGHTS = [1.0, 0.7, 0.9, 1.2, 0.9]
all_preds = []
all_paths = None

for fold in range(CFG.N_FOLDS):
    model_path = CFG.MODEL_DIR / f"best_model_fold{fold}.pth"
    if not model_path.exists():
        print(f"Fold {fold}: skip - not found")
        continue
    
    print(f"\n{'='*40}")
    print(f"Fold {fold}: loading...")
    print(f"{'='*40}")
    
    model = BiomassModel(CFG.BACKBONE, pretrained=False)
    state_dict = torch.load(model_path, map_location="cpu")
    
    # Handle DataParallel wrapper if present
    if list(state_dict.keys())[0].startswith("module."):
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
    
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    
    preds = []
    paths = []
    with torch.no_grad():
        for left, right, p in tqdm(test_loader, desc=f"Fold {fold}"):
            left = left.to(device)
            right = right.to(device)
            with torch.amp.autocast("cuda"):
                out = model((left, right))
            preds.append(out.cpu().numpy())
            paths.extend(p)
    
    preds = np.vstack(preds)
    all_preds.append(preds * FOLD_WEIGHTS[fold])
    if all_paths is None:
        all_paths = paths
    
    print(f"Fold {fold}: done, shape={preds.shape}")
    
    # Clear memory
    del model, state_dict
    torch.cuda.empty_cache()
    gc.collect()

print(f"\n{'='*40}")
print(f"Total folds used: {len(all_preds)}")
print(f"{'='*40}")

In [ ]:
# Ensemble
total_weight = sum(FOLD_WEIGHTS[:len(all_preds)])
ensemble = np.sum(all_preds, axis=0) / total_weight
print(f"Ensemble: {ensemble.shape}")

# Create wide-format predictions DataFrame
preds_wide = pd.DataFrame(ensemble, columns=CFG.TARGETS)
preds_wide.insert(0, 'image_path', all_paths)

print(f"\nPredictions wide format:")
print(preds_wide.head())

# Convert to long format
preds_long = preds_wide.melt(
    id_vars=['image_path'],
    value_vars=CFG.TARGETS,
    var_name='target_name',
    value_name='target'
)

print(f"\nPredictions long format:")
print(preds_long.head(10))

# IMPORTANT: Merge with test_df to get correct sample_id
# test_df has: sample_id, image_path, target_name
print(f"\ntest_df columns: {test_df.columns.tolist()}")
print(f"preds_long columns: {preds_long.columns.tolist()}")

submission = pd.merge(
    test_df[['sample_id', 'image_path', 'target_name']],
    preds_long,
    on=['image_path', 'target_name'],
    how='left'
)

# Keep only required columns
submission = submission[['sample_id', 'target']]

# Check for missing values
missing_count = submission['target'].isna().sum()
if missing_count > 0:
    print(f"\n⚠️ Warning: {missing_count} missing predictions!")
    print(submission[submission['target'].isna()].head())
    submission['target'] = submission['target'].fillna(0.0)
else:
    print("\n✅ All predictions matched!")

# Sort by sample_id (same as working code)
submission = submission.sort_values('sample_id').reset_index(drop=True)

# Clip negative values
submission['target'] = submission['target'].clip(lower=0)

# Save
submission.to_csv("submission.csv", index=False)

print(f"\n✅ Saved: submission.csv")
print(f"Shape: {submission.shape}")
print(f"\nFirst 10 rows:")
print(submission.head(10))
print(f"\nLast 10 rows:")
print(submission.tail(10))
print(f"\nStats:")
print(submission['target'].describe())

# Validation
print(f"\n" + "="*50)
print("VALIDATION")
print("="*50)
print(f"Expected rows: {len(test_df)}")
print(f"Actual rows: {len(submission)}")
print(f"Match: {len(submission) == len(test_df)}")
print(f"No missing: {not submission['target'].isna().any()}")
print(f"Unique IDs: {submission['sample_id'].is_unique}")